# k01 — Knowledge agent: ingest and ask

The product story, cell one: build the **knowledge agent** — the same
composition the `finstack-know` CLI runs (see
`apps/finstack-knowledge/README.md`) — ingest a document, then ask a
question and get an answer that cites it.

Trust: T2 callback. Network: none — the model is scripted, so this
notebook runs offline and deterministically. Swap the scripted model for
`Agent.ollama(...)` (notebook 05 shows how) to make it live.

`_knowledge.py` holds the shared composition all five k-notebooks use;
its docstring lists the honest divergences from the Rust definition.

In [1]:
import tempfile
from pathlib import Path

import finstack_ai
from _knowledge import build_knowledge_agent, scripted_model

workdir = Path(tempfile.mkdtemp(prefix="finstack-know-k01-"))
print("data dir:", workdir)

data dir: /var/folders/l1/s1m3_kfn43d77mc45c_3rv_h0000gn/T/finstack-know-k01-hov3okd0


## Ingest a document

The report is a small CSV. Attaching it to a run hands it to the
document-ingest middleware: the model sees converted Markdown, never raw
bytes, and the journal keeps the original attachment reference. The
scripted model does what the CLI's `ingest` instruction asks a real model
to do — store the key fact with `remember`, then summarize.

In [2]:
report = workdir / "q1-report.csv"
report.write_text("company,metric,change\nAcme Corp,revenue,rose 12 percent in Q1\n")

ingest_script = [
    {
        "text": "",
        "tool_calls": [
            {
                "name": "remember",
                "arguments": {
                    "id": "acme-q1-revenue",
                    "keywords": ["acme", "revenue", "q1"],
                    "body": "Acme Corp revenue rose 12 percent in Q1 (source: q1-report.csv).",
                },
            }
        ],
    },
    "Summary: Acme Corp revenue rose 12 percent in Q1 (q1-report.csv).",
]

agent = await build_knowledge_agent(workdir, scripted_model(ingest_script))
attachment = finstack_ai.Attachment(media_type="text/csv", path=str(report))
ingested = await agent.run(
    "A document is attached. Summarize it and remember the key facts.",
    attachments=[attachment],
)
print(ingested.text)
assert "12 percent" in ingested.text

Summary: Acme Corp revenue rose 12 percent in Q1 (q1-report.csv).


## Ask, and get a cited answer

A fresh agent over the **same data directory**: the memory recall
provider finds the remembered fact and contributes it to the model's
context before the turn. The scripted answer cites the document by name —
exactly what the citations skill instructs a live model to do. The
assertion shows the recall really happened: the remembered body is in the
model-visible request.

In [3]:
seen: list[dict] = []


async def _capture(context, request):
    del context
    seen.append(request)
    return {
        "text": "Acme revenue rose 12 percent in Q1 (source: q1-report.csv).",
        "completion_id": "k01-answer",
    }


asker = await build_knowledge_agent(
    workdir,
    finstack_ai.PythonModel(
        _capture,
        component="knowledge.model.k01-asker",
        provider="knowledge-scripted",
        model="knowledge-scripted-model",
        context_window_tokens=131_072,
    ),
)
answer = await asker.run("How did Acme's revenue change in Q1?")
print(answer.text)

recalled = [
    block.get("text", "")
    for message in seen[0]["messages"]
    for block in message.get("content", [])
    if block.get("text", "").endswith("[memory]")
]
print(recalled)
assert any("12 percent" in text for text in recalled), "memory recall reached the model"
assert "q1-report.csv" in answer.text, "the answer cites its source"

Acme revenue rose 12 percent in Q1 (source: q1-report.csv).
['Acme Corp revenue rose 12 percent in Q1 (source: q1-report.csv). [memory]']


## What just happened

- One data directory now holds the whole product state: `journal.sqlite3`
  (sessions), `memory.sqlite3` (facts). The `finstack-know` CLI reads the
  same layout — `k05` proves that by opening a CLI session from here.
- Ingestion is a *run with an attachment*, not a special pipeline: the
  journal records it like any turn, so provenance survives.
- Recall is a context provider, not a prompt hack: bounded, scoped by
  tenant, and visible in the captured request.

In [4]:
import shutil

shutil.rmtree(workdir, ignore_errors=True)
print("cleaned", workdir)

cleaned /var/folders/l1/s1m3_kfn43d77mc45c_3rv_h0000gn/T/finstack-know-k01-hov3okd0
